# Semana 08 — Banco de Dados Relacionais e Não Relacionais

**Curso:** Análise de Dados com Python — SENAI (Turma T5)
**UC (MSEP):** Manipulação de Dados com Python e SQL (150h) — Bloco 5, Semana 08 (abre o bloco final do módulo)

Esta semana muda de ambiente: até a Semana 07, você trabalhava só com arquivo (CSV, JSON, Excel) e, no máximo, um banco simples de arquivo único (`sqlite3`). A partir de agora, o Restaurante Sabor Caseiro passa a guardar os dados de verdade num banco de dados: você vai instalar o PostgreSQL na sua máquina, conhecer o pgAdmin, e aprender a modelar — antes mesmo de escrever qualquer comando SQL (isso começa na Semana 09) — como as informações do restaurante (clientes, pedidos, filiais, fornecedores) se organizam dentro de um banco.

> Em cada tópico abaixo: **2 exemplos resolvidos** + **1 atividade prática** para você fazer sozinho(a).

### 🧭 De onde você vem: fechando o Bloco 4

Antes de entrar na Semana 08, vale reforçar de onde você está vindo e pra onde está indo dentro do curso.

**O que você viu e estudou na Semana 07:** o mesmo `dre_consolidado.csv` do Restaurante Sabor Caseiro, já limpo desde a Semana 06, virando gráfico com **Matplotlib** (linha, barra, dispersão, customização, subplots e exportação). Você também aprendeu a organizar código em **funções reutilizáveis** e conheceu sua primeira **classe** (`class`, `__init__`, `self`, método), e fechou com um **pipeline ETL** completo — Extract, Transform, Load — carregando o resultado num banco `sqlite3` e consultando com um primeiro `SELECT`.

**O que você fez:** criou gráficos customizados e exportados em `.png`, escreveu uma classe do zero, e montou 2 pipelines ETL de ponta a ponta (um salvando em CSV, outro num banco `sqlite3`), fechando o Mini Projeto do módulo — a Situação de Aprendizagem Integradora que uniu tudo dos Blocos 1 a 4.

**Qual era o objetivo:** te dar as ferramentas pra comunicar um dado já limpo (gráfico) e organizar o código que faz esse trabalho (funções, classes, pipeline) de ponta a ponta.

**Onde você está agora:** encerrando o Módulo Básico e abrindo o **Bloco 5** desta UC — o bloco final, sobre **Bancos de Dados, SQL e integração Python-PostgreSQL**. A Semana 08, que começa agora, resolve a pergunta que ficou em aberto no `sqlite3` da Semana 07: como um banco de dados de verdade guarda e organiza a informação — antes de aprender a consultá-lo com SQL (Semanas 09-10) e integrá-lo ao Python de forma profissional (Semanas 11-12).

---
### 🔄 Retomada — o que vimos na Semana 07

Você já sabe transformar dado limpo em gráfico com Matplotlib, organizar código em funções e classes, e montar um pipeline ETL simples salvando o resultado num banco `sqlite3`. Esta semana usa essa base: o `SELECT * FROM despesas WHERE valor > 30000` que você rodou no fechamento da Semana 07 foi sua primeira consulta SQL — esta semana explica, antes de tudo, COMO um banco de dados guarda a informação que aquele `SELECT` consultou.

---
### 🟢 Abertura — Semana 08: Banco de Dados Relacionais e Não Relacionais

Começa o Bloco 5, o bloco final do curso! Nesta semana você entende <strong>como bancos de dados relacionais e não relacionais organizam a informação</strong>, instala o PostgreSQL e o pgAdmin na sua máquina, e aprende a modelar — no papel e em código — as entidades, atributos e relacionamentos do Restaurante Sabor Caseiro antes de guardar qualquer dado de verdade num banco.

**O que você vai aprender nesta semana:**
- A diferença entre banco relacional e NoSQL, e entre OLTP (operação do dia a dia) e OLAP (análise/relatório)
- Instalar e configurar o PostgreSQL e o pgAdmin, e testar a conexão a partir do Python
- Modelagem conceitual: entidades, atributos, relacionamentos, cardinalidade e o Diagrama Entidade-Relacionamento (DER)
- Modelagem NoSQL: os modelos documento, chave-valor, colunar e grafos

### 📌 Antes de começar: ambiente e dataset desta semana

**Esta semana roda no VS Code local, não no Google Colab.** O Google Colab não mantém um serviço de banco de dados instalado e rodando entre sessões — e a partir de agora você vai conectar o Python a um PostgreSQL de verdade, instalado na sua própria máquina. Se você ainda não tem o VS Code e o Python configurados localmente, retome o `GUIA_GITHUB.md` (raiz do repositório) e o material da Semana 03 antes de continuar.

Esta semana usa um arquivo novo do Restaurante Sabor Caseiro: `pedidos_sabor_caseiro.csv`, com 18 pedidos das 2 filiais já conhecidas (Centro e Zona Sul), com as colunas `id_pedido`, `cliente`, `filial`, `item`, `valor` e `data_hora`. Diferente do `dre_consolidado.csv` (o relatório mensal consolidado que você já conhece, da Semana 06), este arquivo é o tipo de dado que o restaurante gera **a cada venda individual** — e essa diferença é exatamente o assunto da Seção 1.

- `dataset/pedidos_sabor_caseiro.csv` — 18 linhas, um pedido por linha
- `dataset/dre_consolidado.csv` — o mesmo relatório consolidado da Semana 07 (você vai usar de novo, como contraste)

---
## 1. Banco de Dados Relacional x Não Relacional (NoSQL) — e OLTP x OLAP

### O que é, afinal, um banco de dados?

Até agora, todo dado que você usou no curso morava num **arquivo** — um CSV, um JSON, uma planilha Excel — que você abria com `pd.read_csv()` (ou equivalente), lia, e fechava. Um **banco de dados** é um programa diferente: fica **sempre ligado**, rodando em segundo plano, guardando o dado de forma organizada e permitindo que várias pessoas (ou vários programas, ao mesmo tempo) leiam e alterem essa informação sem esbarrar uma na outra — e sem correr o risco de duas pessoas sobrescreverem o mesmo arquivo sem perceber.

É por isso que o Restaurante Sabor Caseiro, que até aqui te mandou um CSV por vez para você limpar e analisar, passa a guardar a informação de verdade num banco: conforme o número de pedidos cresce e mais gente (caixa, cozinha, gerência) precisa consultar e atualizar esse dado ao mesmo tempo, um arquivo solto não aguenta o volume nem o número de pessoas mexendo nele junto. Existem 2 grandes famílias de banco de dados — relacional e não relacional (NoSQL) — que você conhece a seguir.

Um banco de dados **relacional** guarda a informação em **tabelas**: linhas (registros) e colunas (atributos), com uma estrutura fixa definida antes de qualquer dado entrar — o chamado **esquema** (*schema*). PostgreSQL, MySQL e SQL Server são bancos relacionais. Um banco de dados **não relacional** (NoSQL) guarda a informação em outros formatos — documento, chave-valor, colunar ou grafo (você vê os 4 na Seção 4) — geralmente sem exigir um esquema fixo antes de gravar. MongoDB, Redis e Cassandra são bancos NoSQL.

Além do formato, existe outra pergunta importante: para que aquele dado vai ser usado?

| Sigla | Nome completo | Para que serve | Exemplo no Sabor Caseiro |
|---|---|---|---|
| OLTP | *Online Transaction Processing* | Registrar operações do dia a dia, uma de cada vez, rápido | Cada pedido sendo lançado no caixa (`pedidos_sabor_caseiro.csv`) |
| OLAP | *Online Analytical Processing* | Analisar um grande volume de dados já acumulado, para tomar decisão | O relatório mensal consolidado que a diretoria usa (`dre_consolidado.csv`) |

Um sistema OLTP prioriza velocidade de gravação e a exatidão de um registro por vez; um sistema OLAP prioriza consultas agregadas sobre muitos registros de uma vez — a mesma soma com `groupby()` que você já faz desde a Semana 05.

### Bancos analíticos organizam o dado em fato e dimensão

Um banco **OLAP** normalmente separa a informação em 2 tipos de tabela: a **tabela fato** guarda os eventos que você quer medir — números que dá pra somar, contar, tirar média (ex.: cada pedido, com seu `valor`); as **tabelas dimensão** guardam o "quem, o quê, quando" que descreve aquele evento (ex.: o cliente que fez o pedido, o item pedido, o mês em que aconteceu).

| Tabela | Tipo | O que guarda |
|---|---|---|
| `fato_pedidos` | Fato | 1 linha por pedido: `id_cliente` (FK), `id_item` (FK), `id_tempo` (FK), `valor` |
| `dim_cliente` | Dimensão | `id_cliente`, `nome` |
| `dim_item` | Dimensão | `id_item`, `nome_item` |
| `dim_tempo` | Dimensão | `id_tempo`, `data`, `mes`, `ano` |

Esse jeito de organizar (1 tabela fato cercada de tabelas dimensão) se chama **esquema estrela** — é assim que o `dre_consolidado.csv` que você já usou nas Semanas 06 e 07 foi construído por trás: os valores agregados (fato) cruzados com filial/mês/categoria (dimensões).

### 🔹 Exemplo 1 — O mesmo pedido, representado como dado relacional

📖 **Antes do código:** num banco relacional, a informação de um pedido fica separada em 2 tabelas que se relacionam por uma coluna em comum (o **id**): uma tabela `clientes` (1 registro por cliente) e uma tabela `pedidos` (1 registro por pedido, guardando só o `id_cliente`, não o nome do cliente de novo). Isso evita repetir o nome do cliente em toda linha de pedido. `pedidos[["cliente"]].drop_duplicates()` pega só a coluna `cliente` e remove as repetições, sobrando 1 linha por cliente; `.reset_index(drop=True)` renumera o índice do zero (sem essa renumeração, o índice ficaria com "buracos", herdados das linhas removidas), e como esse índice já começa em 0, `clientes.index + 1` cria a coluna `id_cliente` (1, 2, 3...) — o mesmo papel de "código único" que uma chave primária tem num banco relacional. Por fim, `.merge(clientes, on="cliente")` — o mesmo comando de combinar tabelas por uma coluna em comum que você já usa desde a Semana 05 — devolve `id_cliente` para dentro da tabela de pedidos.

In [ ]:
import pandas as pd

pedidos = pd.read_csv("dataset/pedidos_sabor_caseiro.csv")

clientes = pedidos[["cliente"]].drop_duplicates().reset_index(drop=True)
clientes["id_cliente"] = clientes.index + 1
display(clientes)

tabela_pedidos = pedidos.merge(clientes, on="cliente")[["id_pedido", "id_cliente", "filial", "item", "valor", "data_hora"]]
display(tabela_pedidos.head())

### 🔹 Exemplo 2 — O mesmo pedido, representado como documento NoSQL

📖 **Antes do código:** num banco de documentos (o modelo NoSQL mais comum — você vê os outros 3 na Seção 4), a informação de um pedido fica **inteira dentro de um único registro**, sem separar cliente e pedido em 2 tabelas — chamado de dado **desnormalizado**. O módulo `json`, nativo do Python, tem a função `dumps()` ("dump string"), que transforma um dicionário Python num texto formatado JSON — o mesmo formato de arquivo que você já leu na Semana 04; o parâmetro `indent=2` deixa o texto organizado, com 2 espaços de recuo por nível, só para facilitar a leitura, e `ensure_ascii=False` mantém os acentos em vez de trocá-los por código.

In [ ]:
import pandas as pd
import json

pedidos = pd.read_csv("dataset/pedidos_sabor_caseiro.csv")
primeiro_pedido = pedidos.iloc[0]

documento = {
    "id_pedido": int(primeiro_pedido["id_pedido"]),
    "cliente": primeiro_pedido["cliente"],
    "filial": primeiro_pedido["filial"],
    "item": primeiro_pedido["item"],
    "valor": float(primeiro_pedido["valor"]),
    "data_hora": primeiro_pedido["data_hora"],
}

print(json.dumps(documento, indent=2, ensure_ascii=False))

💡 **Relacional x documento, em uma frase:** o modelo relacional evita repetir dado (menos espaço, mais consistência), mas exige juntar tabelas para ver tudo junto (JOIN, Semana 10); o modelo documento já entrega tudo junto (leitura rápida), mas repete o nome do cliente em cada pedido que ele fizer.

![Diagrama comparando o modelo relacional (tabelas) com o modelo documento (JSON aninhado)](img/diagrama_relacional_documento.svg)

### ✏️ Atividade Prática 1 — Sua vez de programar

**Contextualização:** o dono do Restaurante Sabor Caseiro está avaliando dois sistemas para substituir o caderno de pedidos: um usa banco relacional, o outro usa banco de documentos. Antes de decidir, ele pediu para você montar um exemplo de cada um com um pedido real, para comparar lado a lado.

**Comando:** usando `pedidos_sabor_caseiro.csv`, filtre o pedido de `id_pedido` igual a 5 e monte as duas representações: (1) como uma linha da tabela relacional (repita a lógica do Exemplo 1: crie `clientes`, junte com `.merge()` e filtre a linha) e (2) como um documento JSON (repita a lógica do Exemplo 2, usando a linha filtrada).

In [ ]:
import pandas as pd
import json

pedidos = pd.read_csv("dataset/pedidos_sabor_caseiro.csv")

# repetindo a modelagem relacional do Exemplo 1
clientes = pedidos[["cliente"]].drop_duplicates().reset_index(drop=True)
clientes["id_cliente"] = clientes.index + 1
tabela_pedidos = pedidos.merge(clientes, on="cliente")[["id_pedido", "id_cliente", "filial", "item", "valor", "data_hora"]]

pedido_relacional = tabela_pedidos[tabela_pedidos["id_pedido"] == 5]
print("Representação relacional:")
display(pedido_relacional)

# repetindo a modelagem documento do Exemplo 2
linha = pedidos[pedidos["id_pedido"] == 5].iloc[0]
pedido_documento = {
    "id_pedido": int(linha["id_pedido"]),
    "cliente": linha["cliente"],
    "filial": linha["filial"],
    "item": linha["item"],
    "valor": float(linha["valor"]),
    "data_hora": linha["data_hora"],
}
print("\nRepresentação documento:")
print(json.dumps(pedido_documento, indent=2, ensure_ascii=False))

---
## 2. PostgreSQL e pgAdmin — Instalação, Configuração e Primeira Conexão

Até aqui você só instalou **bibliotecas** Python (`pandas`, `openpyxl`...) — bastava um `%pip install` e pronto, dentro do próprio notebook. O PostgreSQL é diferente: é um **programa completo**, separado do Python, que fica rodando sozinho em segundo plano na sua máquina (chamado de **servidor**, porque "serve" pedidos de outros programas) esperando alguém — o pgAdmin, o Python, ou qualquer outro programa — se conectar nele. Se esse programa não estiver rodando, nada mais nesta seção funciona — por isso o primeiro passo é sempre garantir que ele está de pé, fora do notebook, antes de tentar qualquer código.

- **PostgreSQL** — o SGBD (Sistema Gerenciador de Banco de Dados): o programa que guarda e organiza os dados de verdade. Um único PostgreSQL instalado pode hospedar **vários bancos de dados diferentes** ao mesmo tempo — é por isso que ele já vem, por padrão, com um banco chamado `postgres`, e você ainda vai criar um outro, separado, chamado `sabor_caseiro`.
- **pgAdmin** — um programa à parte, com tela e botões, que te deixa ver e mexer no PostgreSQL sem digitar comando nenhum.
- **psycopg2** — a biblioteca Python que serve de ponte entre o seu código e o PostgreSQL (você usa ela mais adiante nesta seção).
- **SQL** — a linguagem usada para pedir e alterar dado dentro de um banco relacional. Você já viu uma prévia dela no fechamento da Semana 07 (`SELECT * FROM despesas WHERE valor > 30000`); aqui você só confirma que consegue "conversar" com o banco, e aprofunda a sintaxe de verdade nas Semanas 09-10.

### Passo a passo — instalação do PostgreSQL (fora do notebook, Windows)

1. **Baixe o instalador oficial** em `https://www.postgresql.org/download/windows/` — clique em "Download the installer" (ele leva ao site da EDB, mantenedora do instalador oficial para Windows). Escolha a versão mais recente disponível.
2. **Rode o instalador** e siga o assistente ("Setup Wizard"). Nas primeiras telas, mantenha marcados os componentes padrão: PostgreSQL Server, pgAdmin 4 e Command Line Tools.
3. Numa das telas, o instalador pede uma **senha para o usuário `postgres`** — o usuário administrador padrão do banco. Anote essa senha com cuidado: você vai digitá-la toda vez que conectar, tanto pelo pgAdmin quanto pelo Python.
4. Mantenha a **porta padrão, `5432`** — é só o número que identifica, na sua máquina, onde o PostgreSQL espera conexões (como um ramal de telefone).
5. Ao final, o instalador pode abrir o **Stack Builder**, oferecendo drivers e ferramentas extras — você não precisa de nada dali para este curso; pode fechar essa janela sem problema.
6. Quando o instalador terminar, o PostgreSQL já fica rodando automaticamente como um **serviço do Windows** — você não precisa "abrir" o PostgreSQL como abre um programa comum. Para confirmar que ele está ativo: abra o menu Iniciar, digite **Serviços** (ou `services.msc`), procure por um item chamado `postgresql-x64-...` na lista, e confira se a coluna Status diz **"Em execução"**. Sempre que a conexão falhar mais adiante, volte aqui primeiro.

> ⚠️ **Confusão comum:** o PostgreSQL vem, por padrão, com um usuário chamado `postgres` **e**, separadamente, com um banco de dados que também já se chama `postgres` — são 2 coisas diferentes com o mesmo nome. Neste curso você não vai usar esse banco `postgres` padrão para nada: no próximo passo, você cria um banco novo, com outro nome (`sabor_caseiro`).

### Primeiro acesso ao pgAdmin e criação do banco `sabor_caseiro`

1. Abra o **pgAdmin 4** (foi instalado junto, no passo 2 acima).
2. **Na primeira vez que abrir**, ele pede uma **Master Password** — uma senha NOVA, exclusiva do próprio pgAdmin (para proteger as senhas que ele vai guardar para você). É diferente da senha do usuário `postgres` que você definiu na instalação — crie uma e guarde; ela só serve para abrir o pgAdmin, não tem nenhuma relação com o Python.
3. No painel esquerdo (o "Object Explorer"), expanda **Servers** → clique no servidor que já vem pré-cadastrado (algo como "PostgreSQL 1x") → digite a senha do usuário `postgres` (a da instalação, não a Master Password) quando for pedida.
4. Com o servidor conectado, clique com o **botão direito em "Databases"** → **Create** → **Database...** → em "Database", digite `sabor_caseiro` → **Save**.
5. **Confira visualmente antes de continuar:** o banco `sabor_caseiro` precisa aparecer na árvore à esquerda, dentro de "Databases". Só depois de ver isso na tela, siga para os exemplos de código abaixo.

> ⚠️ Se a instalação travar ou você não conseguir instalar localmente a tempo, avise o professor — existe uma alternativa de PostgreSQL gratuito na nuvem (Supabase, Neon) para continuar acompanhando a aula.

### 🔹 Exemplo 1 — Testando a conexão com Python

📖 **Antes do código:** `psycopg2` não vem instalada por padrão, por isso a primeira linha instala a variante `psycopg2-binary` (evita ter que compilar nada na sua máquina). `psycopg2.connect(...)` abre uma conexão com o banco, recebendo `host` (endereço do servidor — `localhost` porque o banco está na sua própria máquina), `dbname` (o banco que você criou no pgAdmin — `sabor_caseiro`, não o banco `postgres` padrão), `user` e `password`. `conexao.cursor()` cria um **cursor** — o objeto usado para executar comandos SQL dentro da conexão aberta. `cursor.execute("SELECT version();")` roda esse comando SQL (por enquanto, `SELECT version()` só devolve a versão do PostgreSQL instalado, para confirmar que a conexão funciona), e `cursor.fetchone()` busca a primeira linha do resultado. Repare que o `import psycopg2` está DENTRO do `try`: se você esqueceu de rodar a célula `%pip install` nesta sessão, o Python levanta um `ModuleNotFoundError` bem ali — por isso ele também é capturado, junto com o `OperationalError` que aparece quando o PostgreSQL não está rodando ou algum dado de conexão está errado.

In [ ]:
%pip install psycopg2-binary

try:
    import psycopg2

    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("SELECT version();")
    print("Conectado! Versão do PostgreSQL:", cursor.fetchone()[0])
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão — rode a célula %pip install acima e execute de novo.")
except psycopg2.OperationalError as erro:
    print("Ainda não conectou. Confira, nos Serviços do Windows, se 'postgresql-x64-...' está 'Em execução', e se host/usuário/senha/banco estão corretos.")
    print(f"Erro real do Python: {erro}")

> ⚠️ **Veja como é um erro real do Python.** O `except psycopg2.OperationalError` acima existe porque, sem o PostgreSQL rodando (ou com a senha errada), é exatamente esse erro que aparece — algo como `connection to server at "localhost" (...), port 5432 failed: Connection refused`. Rodar essa célula sem o PostgreSQL instalado ainda é normal: é assim que você confirma, mais tarde, que o setup funcionou — quando a mensagem virar "Conectado!".

### 🔹 Exemplo 2 — Listando as tabelas que já existem no banco

📖 **Antes do código:** todo banco PostgreSQL guarda, automaticamente, um catálogo interno com informação sobre ele mesmo — chamado `information_schema`. A consulta `SELECT table_name FROM information_schema.tables WHERE table_schema = 'public';` pede os nomes de todas as tabelas do "esquema público" — aqui, "esquema" tem um sentido um pouco diferente do "esquema" da Seção 1 (lá era a estrutura fixa de colunas de uma tabela; aqui é mais como uma pasta dentro do banco, que agrupa tabelas — todo banco novo já vem com uma pasta `public` pronta, onde suas tabelas vão morar a partir da Semana 09). Não se preocupe em decorar essa diferença agora. `cursor.fetchall()` busca **todas** as linhas do resultado (diferente do `fetchone()` do Exemplo 1, que busca só a primeira).

In [ ]:
try:
    import psycopg2

    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public';")
    tabelas = cursor.fetchall()
    print("Tabelas encontradas:", tabelas if tabelas else "nenhuma ainda (você cria a primeira na Semana 09)")
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão — rode a célula %pip install do Exemplo 1 primeiro.")
except psycopg2.OperationalError as erro:
    print("Ainda não conectou. Confira, nos Serviços do Windows, se 'postgresql-x64-...' está 'Em execução', e se host/usuário/senha/banco estão corretos.")
    print(f"Erro real do Python: {erro}")

### ✏️ Atividade Prática 2 — Sua vez de programar

**Contextualização:** antes de seguir para a modelagem, o dono do restaurante quer ter certeza de que o ambiente técnico está pronto — sem isso, nenhuma das próximas semanas do Bloco 5 funciona.

**Comando:** instale o PostgreSQL e o pgAdmin na sua máquina, crie o banco `sabor_caseiro` pelo pgAdmin, e rode a célula abaixo (repita o padrão do Exemplo 1) trocando `"SUA_SENHA_AQUI"` pela senha que você definiu na instalação, até ver a mensagem "Conectado!".

In [ ]:
try:
    import psycopg2

    conexao = psycopg2.connect(
        host="localhost",
        dbname="sabor_caseiro",
        user="postgres",
        password="SUA_SENHA_AQUI",
    )
    cursor = conexao.cursor()
    cursor.execute("SELECT version();")
    print("Conectado! Versão do PostgreSQL:", cursor.fetchone()[0])
    conexao.close()
except ModuleNotFoundError:
    print("psycopg2 ainda não foi instalado nesta sessão — rode a célula %pip install do Exemplo 1 primeiro.")
except psycopg2.OperationalError as erro:
    print("Ainda não conectou. Confira, nos Serviços do Windows, se 'postgresql-x64-...' está 'Em execução', e se host/usuário/senha/banco estão corretos.")
    print(f"Erro real do Python: {erro}")

### ✅ Antes de seguir para a Seção 3, confirme:

- [ ] Nos Serviços do Windows (`services.msc`), `postgresql-x64-...` aparece como **"Em execução"**
- [ ] O pgAdmin conecta no servidor sem erro de senha
- [ ] O banco `sabor_caseiro` aparece na árvore do pgAdmin, dentro de "Databases"
- [ ] A célula da Atividade Prática 2 imprimiu **"Conectado! Versão do PostgreSQL: ..."**

Se algum item falhou, volte ao passo a passo de instalação antes de continuar — a Seção 3 não depende do PostgreSQL, mas as Semanas 09-12 inteiras dependem.

---
## 3. Modelagem Conceitual — Entidades, Atributos, Relacionamentos, Cardinalidade e DER

| Termo | O que é | Exemplo no Sabor Caseiro |
|---|---|---|
| Entidade | Um "tipo de coisa" sobre a qual o negócio precisa guardar informação | Cliente, Pedido, Filial |
| Atributo | Uma característica de uma entidade | Cliente tem `nome`, `telefone`; Pedido tem `valor`, `data_hora` |
| Relacionamento | Como 2 entidades se conectam | Um Cliente FAZ um Pedido |
| Cardinalidade | Quantas vezes cada lado do relacionamento pode se repetir | Um Cliente pode fazer VÁRIOS Pedidos |

Cada **atributo** tem um **tipo de dado** — o que aquela coluna pode guardar. Você já usa essa mesma ideia em Python (`int`, `float`, `str`, `bool`); um banco relacional formaliza isso: a coluna só aceita aquele tipo, sempre, porque o esquema (Seção 1) é fixo.

| Tipo de dado | O que guarda | Exemplo no Sabor Caseiro |
|---|---|---|
| Texto | Letras, palavras, texto livre | `nome`, `filial`, `item` |
| Número inteiro | Números sem casa decimal | `id_pedido`, `id_cliente` |
| Número decimal | Números com casa decimal (dinheiro, medidas) | `valor` |
| Data/hora | Uma data e/ou um horário | `data_hora` |
| Booleano | Verdadeiro ou falso, sem meio-termo | *pedido_entregue* (exemplo hipotético — não está no CSV) |

Você declara o tipo de cada coluna na hora de criar a tabela de verdade, no comando `CREATE TABLE` (isso começa na Semana 09) — por enquanto, é só reconhecer que todo atributo tem um tipo, mesmo antes de qualquer SQL.

Existem 3 tipos de cardinalidade:

| Cardinalidade | Significado | Exemplo |
|---|---|---|
| 1:1 | Um registro de A se relaciona com no máximo 1 registro de B | Uma Filial tem 1 endereço |
| 1:N | Um registro de A se relaciona com vários registros de B, mas cada B só pertence a 1 A | Um Cliente faz vários Pedidos; cada Pedido é de 1 só Cliente |
| N:N | Vários registros de A se relacionam com vários registros de B | Uma Filial compra de vários Fornecedores; um Fornecedor vende para várias Filiais |

O **DER** (Diagrama Entidade-Relacionamento) é o desenho que representa entidades (retângulos), atributos e relacionamentos com a cardinalidade marcada — normalmente feito no próprio **pgAdmin, no menu Tools → ERD Tool** (ele desenha o diagrama visualmente, com o mouse), ou numa ferramenta online como o dbdiagram.io, antes de escrever qualquer `CREATE TABLE` (isso começa na Semana 09). Nesta seção, você pratica o raciocínio por trás do DER descrevendo entidades e relacionamentos em código — o mesmo raciocínio que, depois, vira desenho.

### 🔹 Exemplo 1 — Descrevendo as entidades do Sabor Caseiro em código

📖 **Antes do código:** `PK` (chave primária, de *Primary Key*) é o atributo que identifica de forma única cada linha da própria tabela — como o `id_cliente` que você criou na Seção 1. `FK` (chave estrangeira, de *Foreign Key*) é uma coluna que aponta para a PK de outra tabela, criando o relacionamento entre elas — é o `id_cliente` dentro da tabela `Pedido`, apontando de volta para `Cliente`. O código abaixo só organiza essa informação num dicionário Python, um por entidade, e percorre com `for` para imprimir de forma legível.

In [ ]:
entidades = {
    "Cliente": {
        "atributos": ["id_cliente (PK)", "nome", "telefone"],
    },
    "Pedido": {
        "atributos": ["id_pedido (PK)", "id_cliente (FK)", "filial", "item", "valor", "data_hora"],
    },
}

for nome_entidade, detalhes in entidades.items():
    print(f"Entidade: {nome_entidade}")
    for atributo in detalhes["atributos"]:
        print(f"  - {atributo}")

### 🔹 Exemplo 2 — Descrevendo o relacionamento e a cardinalidade

📖 **Antes do código:** o dicionário `relacionamento` guarda as informações que descrevem qualquer relacionamento: quem participa (`entidade_origem`, `entidade_destino`), como o relacionamento se chama no diagrama (`verbo`, no infinitivo em `verbo_infinitivo` — usado só na frase de leitura, para o português ficar correto) e a cardinalidade. O `print()` monta, a partir desses valores, a mesma frase que você diria olhando para um DER desenhado.

In [ ]:
relacionamento = {
    "entidade_origem": "Cliente",
    "entidade_destino": "Pedido",
    "verbo": "FAZ",
    "verbo_infinitivo": "fazer",
    "cardinalidade": "1:N",
}

print(f"{relacionamento['entidade_origem']} (1) --{relacionamento['verbo']}--> (N) {relacionamento['entidade_destino']}")
print(
    f"Leitura: um(a) {relacionamento['entidade_origem']} pode {relacionamento['verbo_infinitivo']} vários(as) "
    f"{relacionamento['entidade_destino']}(s); cada {relacionamento['entidade_destino']} pertence a um(a) único(a) "
    f"{relacionamento['entidade_origem']}."
)

![Diagrama das cardinalidades 1:1, 1:N e N:N, e o DER de Cliente e Pedido](img/diagrama_cardinalidades.svg)![DER simplificado das tabelas Cliente e Pedido, com a chave estrangeira id_cliente ligando as duas](img/diagrama_der_cliente_pedido.svg)

### ✏️ Atividade Prática 3 — Sua vez de programar

**Contextualização:** o restaurante quer passar a registrar também os fornecedores de cada filial — retomando o `fornecedores_despesas.csv` que você limpou na Semana 06. Cada filial compra de vários fornecedores diferentes, e cada fornecedor pode vender para mais de uma filial.

**Comando:** repetindo o formato dos Exemplos 1 e 2, monte (1) as entidades `Filial` (atributos: `id_filial` (PK), `nome`, `endereco`) e `Fornecedor` (atributos: `id_fornecedor` (PK), `nome`, `categoria_despesa`) e (2) o relacionamento entre elas, com a cardinalidade correta.

In [ ]:
entidades = {
    "Filial": {
        "atributos": ["id_filial (PK)", "nome", "endereco"],
    },
    "Fornecedor": {
        "atributos": ["id_fornecedor (PK)", "nome", "categoria_despesa"],
    },
}

for nome_entidade, detalhes in entidades.items():
    print(f"Entidade: {nome_entidade}")
    for atributo in detalhes["atributos"]:
        print(f"  - {atributo}")

relacionamento = {
    "entidade_origem": "Filial",
    "entidade_destino": "Fornecedor",
    "verbo": "COMPRA DE",
    "cardinalidade": "N:N",
}

print(f"\n{relacionamento['entidade_origem']} (N) --{relacionamento['verbo']}--> (N) {relacionamento['entidade_destino']}")
print(
    f"Leitura: uma {relacionamento['entidade_origem']} pode comprar de vários Fornecedores, e um "
    f"{relacionamento['entidade_destino']} pode vender para várias Filiais."
)

---
## 4. Modelagem NoSQL — Documento, Chave-Valor, Colunar e Grafos

| Modelo | Como guarda o dado | Quando usar | Exemplo real |
|---|---|---|---|
| Documento | Um registro inteiro, aninhado, num único arquivo (geralmente JSON) | Registros com estrutura flexível, que mudam de um item para outro | MongoDB |
| Chave-valor | Uma chave única aponta direto para um valor, sem estrutura interna fixa | Buscas muito rápidas por 1 identificador (cache, sessão) | Redis |
| Colunar | Os dados organizados por coluna, não por linha | Agregações rápidas sobre muitos registros (somar, contar) | Cassandra |
| Grafo | Nós (entidades) e conexões (relacionamentos) entre eles | Quando o que importa é a rede de relacionamentos em si | Neo4j |

MongoDB, Redis, Cassandra e Neo4j não são conceitos abstratos — são **produtos de banco de dados reais**, cada um especializado num desses 4 modelos, usados por empresas de todos os tamanhos no mercado (do mesmo jeito que PostgreSQL e MySQL são produtos reais de banco relacional).

### 🔹 Exemplo 1 — Documento e Chave-Valor

📖 **Antes do código:** `.to_dict()`, chamado numa linha de um DataFrame (uma `Series`), converte aquela linha num dicionário Python — a mesma ideia do documento da Seção 1, só que usando um método pronto do Pandas em vez de montar o dicionário campo a campo. Já o dicionário `cache_pedidos` é um exemplo de chave-valor: a expressão `{chave: valor for ...}` é uma **dict comprehension** — o mesmo padrão de list comprehension que você já usa desde o Bloco 2, só que construindo um dicionário. `pedidos.iterrows()` percorre o DataFrame linha por linha, devolvendo, a cada volta, o índice da linha (aqui ignorado, por isso o nome `_`) e a linha em si.

In [ ]:
import pandas as pd
import json

pedidos = pd.read_csv("dataset/pedidos_sabor_caseiro.csv")

# Documento: um pedido inteiro, aninhado
pedido_documento = pedidos.iloc[2].to_dict()
print("Documento:")
print(json.dumps(pedido_documento, indent=2, ensure_ascii=False))

# Chave-valor: só o essencial, para consulta rápida por id
cache_pedidos = {int(linha["id_pedido"]): linha["item"] for _, linha in pedidos.iterrows()}
print("\nChave-valor (id do pedido → item):")
print(cache_pedidos)

### 🔹 Exemplo 2 — Colunar e Grafos

📖 **Antes do código:** `.to_dict(orient="list")`, chamado num DataFrame inteiro (diferente do `.to_dict()` do Exemplo 1, que era numa linha só), monta 1 lista por coluna — os valores de `item` todos juntos, os valores de `valor` todos juntos — em vez de 1 dicionário por linha. É exatamente essa ideia de "organizar por coluna" que um banco colunar usa para somar/agregar rápido sobre milhões de registros, sem precisar ler as outras colunas. Já `grafo_indicacoes` é um dado **inventado, só para ilustrar o formato**: cada cliente aponta para uma lista de clientes que ele indicou — a mesma estrutura (nó → lista de conexões) que um banco de grafos usa para guardar redes de relacionamento.

![Diagrama dos 4 modelos NoSQL: documento, chave-valor, colunar e grafo](img/diagrama_4_modelos_nosql.svg)

In [ ]:
import pandas as pd

pedidos = pd.read_csv("dataset/pedidos_sabor_caseiro.csv")

# Colunar: os dados organizados por coluna, não por linha
armazenamento_colunar = pedidos[["item", "valor"]].to_dict(orient="list")
print("Colunar (por coluna):")
print(armazenamento_colunar)

# Grafo: quem indicou quem (dado inventado, só para ilustrar o formato)
grafo_indicacoes = {
    "Marcos Silva": ["Juliana Alves"],
    "Juliana Alves": ["Ricardo Nunes", "Fernanda Costa"],
    "Ricardo Nunes": [],
    "Fernanda Costa": ["Patrícia Gomes"],
}
print("\nGrafo (quem indicou quem):")
for cliente, indicados in grafo_indicacoes.items():
    print(f"{cliente} indicou: {indicados if indicados else 'ninguém ainda'}")

### ✏️ Atividade Prática 4 — Sua vez de programar

**Contextualização:** o time de tecnologia do restaurante quer avaliar se vale a pena guardar o cardápio num banco de documentos — porque alguns itens têm atributos que outros não têm (ex.: só a Marmita Executiva tem opção de tamanho).

**Comando:** monte uma lista de dicionários Python representando o cardápio como documentos — inclua pelo menos 3 itens, e faça pelo menos 1 deles ter uma chave a mais que os outros (mostrando a flexibilidade do modelo documento, que não exige o mesmo esquema em todo registro). Imprima o resultado com `json.dumps(..., indent=2, ensure_ascii=False)`.

In [ ]:
import json

cardapio = [
    {"item": "Marmita Executiva", "valor": 32.90, "tamanho": "único"},
    {"item": "Feijoada Completa", "valor": 45.00},
    {"item": "Suco Natural", "valor": 9.50, "sabores": ["Laranja", "Abacaxi", "Maracujá"]},
]

print(json.dumps(cardapio, indent=2, ensure_ascii=False))

---
## 5. Treino em Squads — Sexta-feira (Encontro 3)

Esta seção é usada **em sala (ou em salas remotas/breakout)** na sexta-feira. Cada squad recebe um cenário de negócio diferente, com uma amostra de registros que esse negócio precisa guardar. O desafio: (1) decidir se o cenário pede um banco **relacional** ou **NoSQL**, justificando, e (2) modelar as entidades, atributos e o relacionamento principal (com cardinalidade) — no mesmo formato dos Exemplos 1 e 2 da Seção 3. Repare que cada squad, propositalmente, tem uma cardinalidade diferente (1:N, N:N ou 1:1) — parte do desafio é identificar qual.

### Squad B — Loja de Roupas Online

**Contextualização:** a loja quer digitalizar o controle de pedidos, hoje anotado em papel — cada cliente pode fazer vários pedidos ao longo do tempo, e a loja precisa saber exatamente quais peças cada pedido contém.

In [ ]:
amostra_squad_b = [
    {"cliente": "Beatriz Lima", "pedido": "Camiseta P", "valor": 59.90},
    {"cliente": "Beatriz Lima", "pedido": "Calça Jeans M", "valor": 129.90},
    {"cliente": "Diego Rocha", "pedido": "Jaqueta G", "valor": 219.90},
]

# decida: relacional ou NoSQL? modele as entidades, atributos e a cardinalidade do relacionamento

### Squad C — Plataforma de Streaming de Música

**Contextualização:** a plataforma quer estruturar como playlists e músicas se relacionam, sabendo que a mesma música pode aparecer em várias playlists diferentes, e uma playlist reúne várias músicas — repare que `Lo-fi Beats 01` aparece em 2 playlists na amostra abaixo.

In [ ]:
amostra_squad_c = [
    {"playlist": "Foco no Trabalho", "musica": "Lo-fi Beats 01"},
    {"playlist": "Foco no Trabalho", "musica": "Lo-fi Beats 02"},
    {"playlist": "Relaxar à Noite", "musica": "Lo-fi Beats 01"},
    {"playlist": "Treino", "musica": "Corrida Intensa"},
]

# decida: relacional ou NoSQL? modele as entidades Playlist e Música, e a cardinalidade do relacionamento

### Squad D — Clínica Médica

**Contextualização:** a clínica quer digitalizar o prontuário dos pacientes — cada paciente tem exatamente 1 prontuário, com um número único que não pode se repetir nem se perder.

In [ ]:
amostra_squad_d = [
    {"paciente": "Otávio Ramos", "numero_prontuario": "PRT-1042", "tipo_sanguineo": "O+"},
    {"paciente": "Vanessa Melo", "numero_prontuario": "PRT-1043", "tipo_sanguineo": "A-"},
]

# decida: relacional ou NoSQL? modele as entidades Paciente e Prontuário, e a cardinalidade do relacionamento
# (repare que cada paciente tem só 1 prontuário, e cada prontuário pertence a só 1 paciente)

### Squad E — Aplicativo de Delivery

**Contextualização:** o aplicativo quer entender como usuários e restaurantes se conectam através dos pedidos — um usuário pode pedir de vários restaurantes diferentes, e um restaurante atende vários usuários diferentes.

In [ ]:
amostra_squad_e = [
    {"usuario": "Renata Duarte", "restaurante": "Pizzaria Napoli", "valor": 68.00},
    {"usuario": "Renata Duarte", "restaurante": "Sabor Caseiro - Centro", "valor": 32.90},
    {"usuario": "Thiago Nogueira", "restaurante": "Sabor Caseiro - Centro", "valor": 45.00},
    {"usuario": "Thiago Nogueira", "restaurante": "Pizzaria Napoli", "valor": 52.00},
]

# decida: relacional ou NoSQL? modele as entidades Usuário e Restaurante, e a cardinalidade do relacionamento

### 🗣️ Debate coletivo (após as apresentações)

Depois que todos os squads apresentarem, discuta com a turma:

- Repare que os 4 squads tiveram cardinalidades diferentes (Squad B: 1:N; Squads C e E: N:N; Squad D: 1:1) — algum squad ficou em dúvida sobre qual cardinalidade o próprio cenário pedia?
- Algum squad ficou em dúvida entre relacional e NoSQL? O que pesou na decisão final?
- Se o cenário do seu squad crescesse (milhões de registros), a decisão relacional x NoSQL mudaria?

---
### 🏁 Fechamento — Semana 08

**Nesta semana você aprendeu:**
- A diferença entre banco relacional e NoSQL, e entre OLTP e OLAP
- Instalar e configurar o PostgreSQL e o pgAdmin, testando a conexão a partir do Python
- Modelar entidades, atributos, relacionamentos e cardinalidade — a base de um DER
- Os 4 modelos de banco NoSQL: documento, chave-valor, colunar e grafo

**Próxima semana:** com o modelo pronto na cabeça, a Semana 09 transforma tudo isso em tabelas de verdade — normalização, chaves primárias/estrangeiras, e os primeiros comandos SQL (`CREATE TABLE`, `INSERT`, `SELECT`).

---
### Assinatura

Curso: **Análise de Dados com Python — SENAI (Turma T5)**
Semana 08 — Banco de Dados Relacionais e Não Relacionais

*Prof. Especialista Cláudio F. Neves*